# RNAAPIA — Architecture Search: FaceCNN (v10)

**Objetivo:** encontrar a melhor arquitetura para o FaceCNN usando Optuna:
- Número de blocos convolucionais (2 → 6)
- Canais por bloco
- Camadas densas intermédias (0 → 2)
- Embedding size

**Hiperparâmetros fixos** (melhores do v9):
- LR: 1.47e-4 | Weight decay: 6.5e-5 | Label smoothing: 0.071 | Batch: 64 | Scheduler: plateau

**Referência v9:** val 96.06% | test 96.24%

```
0. Setup
1. Dataset + MTCNN
2. FaceCNN Dinâmico
3. Funções auxiliares
4. Objective Function
5. Optuna Study
6. Análise dos Resultados
7. Treino Final
8. Guardar Resultados
```

## ⚠️ Antes de começar
1. **Add Data** → `hearfool/vggface2`
2. **Add Data** → Models → `facecnn` (pesos v9)
3. **Add Data** → `dataset-metadata-json`
4. **Settings** → Accelerator → GPU T4

---
## 0. Setup

In [ ]:
# Carrega bibliotecas, ativa configurações do ambiente e prepara o uso do Optuna.
# Como esta experiência testa muitas arquiteturas, a organização inicial do ambiente é essencial.
import torch, os, glob, json, time, warnings, shutil
warnings.filterwarnings('ignore')

import torch.nn as nn
import torch.nn.functional as F
import optuna
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from copy import deepcopy
optuna.logging.set_verbosity(optuna.logging.WARNING)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'CUDA: {torch.cuda.is_available()}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
%%capture
# Instala os pacotes necessários para pesquisa arquitetural, alinhamento facial e métricas.
# O output fica oculto para deixar o notebook mais limpo e legível.
!pip install optuna facenet-pytorch==2.5.3 tqdm scikit-learn matplotlib
print('✓ Dependências instaladas')

In [ ]:
# Define os diretórios usados durante a pesquisa arquitetural e cria-os quando necessário.
# Isto separa claramente checkpoints, estudos Optuna e logs da experiência v10.
BASE  = '/kaggle/working'
PATHS = {
    'raw'        : '/kaggle/input/datasets/hearfool/vggface2/train',
    'aligned'    : f'{BASE}/vggface2_aligned',
    'checkpoints': f'{BASE}/checkpoints',
    'optuna'     : f'{BASE}/optuna_arch',
    'logs'       : f'{BASE}/logs',
}
for k, v in PATHS.items():
    if k != 'raw':
        os.makedirs(v, exist_ok=True)

# Pesos v9
CNN_CKPT_IN = '/kaggle/input/models/goncalojesus/facecnn/pytorch/default/1/best_cnn.pth'

# Metadata
METADATA_PATH = None
for candidate in [
    '/kaggle/input/datasets/goncalojesus/dataset-metadata-v7-json/dataset_metadata_v7.json',
    '/kaggle/input/datasets/goncalojesus/dataset-metadata-json/dataset_metadata.json',
]:
    if os.path.exists(candidate):
        METADATA_PATH = candidate
        print(f'✓ Metadata: {candidate}')
        break

# Configuração
IMAGE_SIZE        = 192
N_TRIALS          = 30
EPOCHS_TRIAL      = 12
EPOCHS_FINAL      = 60
MAX_PARAMS        = 50_000_000

# Hiperparâmetros fixos do v9
BEST_LR           = 0.000147
BEST_WEIGHT_DECAY = 0.000065
BEST_LABEL_SMOOTH = 0.071351
BEST_BATCH_SIZE   = 64

# Referência v9
V9_VAL_ACC  = 96.06
V9_TEST_ACC = 96.24

print(f'N_TRIALS={N_TRIALS} | EPOCHS_TRIAL={EPOCHS_TRIAL} | EPOCHS_FINAL={EPOCHS_FINAL}')
print(f'Referência v9: val {V9_VAL_ACC}% | test {V9_TEST_ACC}%')

---
## 1. Dataset + MTCNN

In [ ]:
# Reaproveita o dataset alinhado com MTCNN ou prepara esse passo se ainda estiver em falta.
# O objetivo é garantir que todas as arquiteturas competem sobre o mesmo conjunto de dados.
from facenet_pytorch import MTCNN
from PIL import Image
from tqdm import tqdm

all_identities = sorted(os.listdir(PATHS['raw']))

# Carregar identidades do metadata
if METADATA_PATH and os.path.exists(METADATA_PATH):
    with open(METADATA_PATH) as f:
        metadata = json.load(f)
    identities_to_use = metadata['identities']
    print(f'✓ {len(identities_to_use)} identidades do metadata')
else:
    identities_to_use = all_identities[:50]
    print(f'Sem metadata — a usar primeiras 50 identidades')

# Inicializar MTCNN — mesmo setup dos notebooks anteriores
margin = max(20, IMAGE_SIZE // 6)
mtcnn  = MTCNN(
    image_size=IMAGE_SIZE,
    margin=margin,
    min_face_size=20,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=False,   # devolve tensor uint8 [0,255], não normalizado
    device=DEVICE
)

already_done = set(os.listdir(PATHS['aligned'])) if os.path.exists(PATHS['aligned']) else set()
remaining    = [i for i in identities_to_use if i not in already_done]
print(f'Já alinhadas: {len(already_done)} | Por alinhar: {len(remaining)}')

total_saved, failed = 0, 0
for identity in tqdm(remaining, desc='MTCNN'):
    out_dir   = os.path.join(PATHS['aligned'], identity)
    os.makedirs(out_dir, exist_ok=True)
    img_paths = (glob.glob(os.path.join(PATHS['raw'], identity, '*.jpg')) +
                 glob.glob(os.path.join(PATHS['raw'], identity, '*.png')))[:350]
    saved = 0
    for p in img_paths:
        try:
            face = mtcnn(Image.open(p).convert('RGB'))
            if face is not None:
                # face é tensor uint8 [C,H,W] — converter para PIL e guardar
                Image.fromarray(face.permute(1,2,0).byte().numpy()).save(
                    os.path.join(out_dir, os.path.basename(p)))
                saved += 1
        except Exception:
            failed += 1
    if saved < 20:
        shutil.rmtree(out_dir)
    else:
        total_saved += saved

total_imgs = sum(len(os.listdir(os.path.join(PATHS['aligned'], d)))
                 for d in os.listdir(PATHS['aligned']))
print(f'\n✓ MTCNN: {len(os.listdir(PATHS["aligned"]))} identidades | {total_imgs} imagens | {failed} falhas')

In [ ]:
# Declara as transformações de treino, validação e teste usadas ao longo da pesquisa.
# O augmentation mantém robustez visual sem introduzir diferenças injustas entre trials.
aug_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.1),
    transforms.RandomApply([transforms.RandomRotation(15)], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
])
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

full_dataset = datasets.ImageFolder(PATHS['aligned'], transform=aug_transform)
NUM_CLASSES  = len(full_dataset.classes)
total        = len(full_dataset)
n_train = int(0.70 * total)
n_val   = int(0.15 * total)
n_test  = total - n_train - n_val

train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

def get_loaders(batch_size):
    aug_ds = datasets.ImageFolder(PATHS['aligned'], transform=aug_transform)
    val_ds = datasets.ImageFolder(PATHS['aligned'], transform=val_transform)
    return (
        DataLoader(Subset(aug_ds, train_set.indices), batch_size=batch_size,
                   shuffle=True,  num_workers=2, pin_memory=True),
        DataLoader(Subset(val_ds, val_set.indices),   batch_size=batch_size,
                   shuffle=False, num_workers=2, pin_memory=True),
        DataLoader(Subset(val_ds, test_set.indices),  batch_size=batch_size,
                   shuffle=False, num_workers=2, pin_memory=True),
    )

print(f'Classes: {NUM_CLASSES} | Train: {n_train} | Val: {n_val} | Test: {n_test}')

---
## 2. FaceCNN Dinâmico

In [ ]:
# Implementa a versão dinâmica da FaceCNN, cuja profundidade e largura podem variar por trial.
# Esta flexibilidade permite ao Optuna explorar arquiteturas mais ricas do que a baseline fixa.
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),  # Extrai padrões visuais locais da face.
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.MaxPool2d(2))  # Normaliza, ativa e reduz resolução dentro do bloco.
    def forward(self, x): return self.block(x)


class DynamicFaceCNN(nn.Module):
    """
    FaceCNN com arquitetura parametrizável.

    conv_channels: lista de canais (e.g. [32,64,128,256]) — len = nº de blocos
    dense_sizes:   lista de camadas densas intermédias (e.g. [512] ou [])
    embedding_size: dimensão final do embedding

    Com IMAGE_SIZE=192 e N blocos conv:
      N=2 → fm=48 | N=3 → fm=24 | N=4 → fm=12 | N=5 → fm=6 | N=6 → fm=3
    """
    def __init__(self, num_classes, conv_channels, dense_sizes,
                 embedding_size, image_size=192, dropout=0.5):
        super().__init__()

        # Blocos convolucionais
        conv_layers, in_ch = [], 3
        for out_ch in conv_channels:
            conv_layers.append(ConvBlock(in_ch, out_ch))  # Cada bloco aumenta a capacidade representacional da rede.
            in_ch = out_ch
        self.conv_blocks = nn.Sequential(*conv_layers)

        fm        = image_size // (2 ** len(conv_channels))  # Tamanho espacial final após todos os poolings.
        flat_size = conv_channels[-1] * fm * fm  # Número de features que entram na cabeça densa.

        # Camadas densas
        dense_layers = [nn.Flatten(), nn.Dropout(p=dropout)]  # A cabeça densa é montada dinamicamente pelo trial.
        in_feat = flat_size
        for d_size in dense_sizes:
            dense_layers += [
                nn.Linear(in_feat, d_size),  # Camada totalmente ligada intermédia opcional.
                nn.BatchNorm1d(d_size),
                nn.ReLU(inplace=True),
            ]
            in_feat = d_size
        dense_layers += [
            nn.Linear(in_feat, embedding_size),  # Produz o embedding final comparável entre identidades.
            nn.BatchNorm1d(embedding_size),
            nn.ReLU(inplace=True),
        ]
        self.embedding  = nn.Sequential(*dense_layers)
        self.classifier = nn.Linear(embedding_size, num_classes)

    def forward(self, x, return_embedding=False):
        emb = self.embedding(self.conv_blocks(x))  # Forward completo: conv -> cabeça densa -> embedding.
        return F.normalize(emb, dim=1) if return_embedding else self.classifier(emb)

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Sanity check
test_model = DynamicFaceCNN(
    num_classes=NUM_CLASSES,
    conv_channels=[32, 64, 128, 256],
    dense_sizes=[],
    embedding_size=512,
    image_size=IMAGE_SIZE,
    dropout=0.5
).to(DEVICE)
x_dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
out = test_model(x_dummy)
print(f'✓ Sanity check: saída {out.shape} | parâmetros: {test_model.count_params():,}')
del test_model, x_dummy, out
torch.cuda.empty_cache()

---
## 3. Funções Auxiliares

In [ ]:
# Reúne funções auxiliares para carregar pesos compatíveis, treinar e avaliar arquiteturas variáveis.
# A carga parcial aproveita conhecimento prévio mesmo quando a arquitetura muda ligeiramente.
def load_partial_weights(model, ckpt_path):
    """Carrega pesos compatíveis do checkpoint — ignora camadas com dimensão diferente."""
    if not ckpt_path or not os.path.exists(ckpt_path):
        return 0
    state      = torch.load(ckpt_path, map_location=DEVICE)['model']  # Recupera pesos do melhor modelo conhecido.
    model_dict = model.state_dict()
    to_load    = {k: v for k, v in state.items()
                  if k in model_dict and 'classifier' not in k
                  and v.shape == model_dict[k].shape}
    model_dict.update(to_load)
    model.load_state_dict(model_dict)
    return len(to_load)


def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()  # Ativa comportamento de treino para dropout e batch norm.
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()  # Evita acumular gradientes entre batches.
        with torch.amp.autocast('cuda'):
            out  = model(xb)  # Forward da arquitetura candidata.
            loss = criterion(out, yb)  # Loss de classificação para o batch atual.
        scaler.scale(loss).backward()  # Backward em mixed precision.
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * xb.size(0)
        correct    += (out.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()  # Modo de avaliação para medir generalização de forma consistente.
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        out  = model(xb).float()
        loss = criterion(out, yb)
        if not torch.isnan(loss):
            total_loss += loss.item() * xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        total   += xb.size(0)
    return total_loss / total, correct / total


print('✓ Funções prontas')

---
## 4. Objective Function — Optuna
Espaço de pesquisa:
- Blocos conv: 2-6 | Canais: crescentes por bloco
- Camadas densas intermédias: 0-2
- Embedding size: 256 / 512 / 1024
- Dropout: 0.2-0.6

In [ ]:
# Define o espaço de pesquisa da arquitetura e a função que traduz sugestões do trial em camadas reais.
# Aqui o Optuna decide quantos blocos usar, quantos canais escolher e como montar a cabeça densa.
CONV_CHOICES = {
    1: [16, 32, 64],
    2: [32, 64, 128],
    3: [64, 128, 256],
    4: [128, 256, 512],
    5: [256, 512],
    6: [256, 512],
}

# Nesta função, o Optuna define a própria arquitetura da rede.
# Em vez de escolher só hiperparâmetros de treino, cada trial decide como a CNN será construída.
def suggest_architecture(trial):
    # O número de blocos convolucionais é um dos elementos mais importantes do architecture search.
    # Mais blocos podem aumentar capacidade, mas também custo e risco de overfitting.
    n_conv   = trial.suggest_int('n_conv_blocks', 2, 6)  # O trial decide a profundidade da CNN.
    channels = [trial.suggest_categorical(f'conv_ch_{i}', CONV_CHOICES[i])
                for i in range(1, n_conv + 1)]
    # Garantir canais crescentes
    for i in range(1, len(channels)):
        if channels[i] < channels[i-1]:
            channels[i] = channels[i-1]

    # O trial também decide quantas camadas densas intermédias devem existir antes do embedding final.
    n_dense    = trial.suggest_int('n_dense_layers', 0, 2)
    dense_list = []
    if n_dense >= 1:
        dense_list.append(trial.suggest_categorical('dense_size_1', [256, 512, 1024]))
    if n_dense >= 2:
        dense_list.append(trial.suggest_categorical('dense_size_2', [128, 256, 512]))

    emb_size = trial.suggest_categorical('embedding_size', [256, 512, 1024])
    dropout  = trial.suggest_float('dropout', 0.2, 0.6)

    return channels, dense_list, emb_size, dropout


# Esta objective traduz a arquitetura sugerida pelo trial num modelo real,
# treina-o por algumas épocas e devolve a melhor val_acc observada.
def objective(trial):
    # Primeiro o trial escolhe a arquitetura concreta: profundidade, canais, cabeça densa e dropout.
    channels, dense_list, emb_size, dropout = suggest_architecture(trial)

    model = DynamicFaceCNN(
        num_classes=NUM_CLASSES,
        conv_channels=channels,
        dense_sizes=dense_list,
        embedding_size=emb_size,
        image_size=IMAGE_SIZE,
        dropout=dropout,
    ).to(DEVICE)

    # Antes de treinar, rejeitamos arquiteturas demasiado grandes para o orçamento computacional definido.
    if model.count_params() > MAX_PARAMS:
        del model
        raise optuna.exceptions.TrialPruned()  # Descarta arquiteturas demasiado pesadas para o orçamento definido.

    # Sempre que possível, carregamos pesos compatíveis do melhor modelo anterior.
    # Isto acelera a adaptação de novas arquiteturas e reduz o custo da pesquisa.
    n_loaded = load_partial_weights(model, CNN_CKPT_IN)

    train_loader, val_loader, _ = get_loaders(BEST_BATCH_SIZE)
    criterion = nn.CrossEntropyLoss(label_smoothing=BEST_LABEL_SMOOTH)  # Mantém a mesma loss do melhor pipeline anterior.
    optimizer = Adam(model.parameters(), lr=BEST_LR, weight_decay=BEST_WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=3,
                                  factor=0.5, min_lr=1e-6)  # Se a validação estagnar, o LR baixa.
    scaler    = torch.amp.GradScaler('cuda')

    # Tal como na hiperparametrização, cada trial é avaliado pela melhor accuracy de validação obtida.
    best_val_acc = 0.0
    for epoch in tqdm(range(EPOCHS_TRIAL),
                      desc=f'Trial {trial.number} {channels}', leave=False):
        train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        _, val_acc = evaluate(model, val_loader, criterion)
        scheduler.step(val_acc)

        # O Optuna recebe feedback intermédio desta arquitetura ao longo das épocas.
        trial.report(val_acc * 100, epoch)  # Reporta a métrica intermédia ao Optuna.
        # Se a arquitetura estiver a ter desempenho fraco, o trial é interrompido cedo.
        if trial.should_prune():
            del model; torch.cuda.empty_cache()
            raise optuna.exceptions.TrialPruned()

        best_val_acc = max(best_val_acc, val_acc)

    del model; torch.cuda.empty_cache()
    # O valor devolvido representa a qualidade final desta arquitetura dentro da study.
    return best_val_acc * 100


print('✓ Objective function definida')

---
## 5. Optuna Study — Architecture Search

In [ ]:
# Cria ou retoma o estudo Optuna responsável por procurar a melhor arquitetura da FaceCNN.
# O notebook também calcula quantos trials ainda faltam para cumprir o orçamento definido.
# Esta study controla toda a pesquisa arquitetural da DynamicFaceCNN.
# Ao contrário da study anterior, aqui o Optuna está a otimizar a estrutura da rede e não apenas o treino.
study = optuna.create_study(
    direction  = 'maximize',
    # O TPESampler começa por explorar algumas arquiteturas iniciais e depois concentra-se nas mais promissoras.
    sampler    = optuna.samplers.TPESampler(seed=42, n_startup_trials=8),
    # O pruning é ainda mais importante aqui, porque testar arquiteturas ruins pode ser muito caro.
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    study_name = 'FaceCNN_arch',
    storage    = f'sqlite:///{PATHS["optuna"]}/arch_study.db',
    load_if_exists = True
)

# Se a base de dados já existir, a architecture search pode continuar do ponto onde ficou.
already_done = len(study.trials)
remaining    = max(0, N_TRIALS - already_done)
print(f'Trials já concluídos: {already_done} | Por correr: {remaining}')

if remaining > 0:
    # O método optimize executa sucessivos trials, cada um com uma arquitetura diferente.
    study.optimize(
        objective,
        n_trials = remaining,
        timeout  = 7200,  # 2 horas
        callbacks=[lambda study, trial: print(
            f'Trial {trial.number:02d} | '
            f'{trial.value:.2f}% | '
            f'conv={trial.params.get("n_conv_blocks")}blocos | '
            f'dense={trial.params.get("n_dense_layers")} | '
            f'emb={trial.params.get("embedding_size")} | '
            f'do={trial.params.get("dropout", 0):.2f}'
        ) if trial.value is not None else print(f'Trial {trial.number:02d} | PRUNED')]
    )

# No final, best_trial contém a arquitetura vencedora e a sua pontuação.
best = study.best_trial
print(f'\n✓ Study concluído!')
print(f'  Melhor val acc: {best.value:.2f}%')
print(f'  vs v9:          {best.value - V9_VAL_ACC:+.2f}%')
print(f'  Parâmetros:')
for k, v in best.params.items():
    print(f'    {k}: {v}')

---
## 6. Análise dos Resultados

In [ ]:
# Resume e visualiza os melhores trials para entender que tipo de arquitetura está a funcionar melhor.
# Esta leitura é importante porque o melhor resultado nem sempre é a arquitetura mais simples.
import matplotlib.pyplot as plt
import pandas as pd

complete = sorted([t for t in study.trials if t.value is not None],
                  key=lambda t: t.value, reverse=True)

# Top 5
print('Top 5 arquiteturas:')
print(f'{"#":>4} {"Val Acc":>8} {"Conv":>5} {"Canais":<25} {"Dense":>5} {"Emb":>5} {"Dropout":>8} {"Params":>10}')
print('-' * 80)
for t in complete[:5]:
    p  = t.params
    nc = p['n_conv_blocks']
    ch = [p[f'conv_ch_{i}'] for i in range(1, nc+1)]
    nd = p['n_dense_layers']
    ds = [p.get(f'dense_size_{i}', '') for i in range(1, nd+1)]
    model_tmp = DynamicFaceCNN(
        num_classes=NUM_CLASSES,
        conv_channels=ch, dense_sizes=[d for d in ds if d],
        embedding_size=p['embedding_size'],
        image_size=IMAGE_SIZE, dropout=p['dropout']
    )
    nparams = model_tmp.count_params()
    del model_tmp
    print(f'{t.number:>4} {t.value:>7.2f}% {nc:>5} {str(ch):<25} {nd:>5} '
          f'{p["embedding_size"]:>5} {p["dropout"]:>8.2f} {nparams:>10,}')

# Gráfico
vals = [t.value for t in study.trials if t.value is not None]
best_so_far = [max(vals[:i+1]) for i in range(len(vals))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(range(len(vals)), vals, alpha=0.5, s=30, label='Trial')
axes[0].plot(range(len(best_so_far)), best_so_far, 'r-', linewidth=2, label='Melhor')
axes[0].axhline(V9_VAL_ACC, color='blue', linestyle='--', alpha=0.5, label=f'v9: {V9_VAL_ACC}%')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('Val Accuracy (%)')
axes[0].set_title('Optimization History'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

try:
    importances = optuna.importance.get_param_importances(study)
    axes[1].barh(list(importances.keys()), list(importances.values()), color='#0077b6', alpha=0.8)
    axes[1].set_xlabel('Importância'); axes[1].set_title('Importância dos Parâmetros')
    axes[1].grid(True, alpha=0.3, axis='x')
except Exception:
    axes[1].text(0.5, 0.5, 'Dados insuficientes', ha='center', va='center', transform=axes[1].transAxes)

plt.suptitle('Optuna — Architecture Search FaceCNN', fontsize=13)
plt.tight_layout()
plt.savefig(f'{BASE}/arch_search.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Treino Final com Melhor Arquitetura

In [ ]:
# Extrai os melhores parâmetros arquiteturais e constrói o modelo final com essa configuração.
# A partir daqui deixamos a fase exploratória e passamos ao treino definitivo.
bp = best.params
n_conv = bp['n_conv_blocks']
ch_best = [bp[f'conv_ch_{i}'] for i in range(1, n_conv+1)]
n_dense = bp['n_dense_layers']
ds_best = [bp.get(f'dense_size_{i}') for i in range(1, n_dense+1) if bp.get(f'dense_size_{i}')]
emb_best = bp['embedding_size']
do_best  = bp['dropout']

model_final = DynamicFaceCNN(
    num_classes=NUM_CLASSES,
    conv_channels=ch_best,
    dense_sizes=ds_best,
    embedding_size=emb_best,
    image_size=IMAGE_SIZE,
    dropout=do_best,
).to(DEVICE)

n_loaded = load_partial_weights(model_final, CNN_CKPT_IN)
print(f'Arquitetura final:')
print(f'  Conv blocks: {ch_best}')
print(f'  Dense sizes: {ds_best}')
print(f'  Embedding:   {emb_best}')
print(f'  Dropout:     {do_best:.3f}')
print(f'  Parâmetros:  {model_final.count_params():,}')
print(f'  Pesos carregados do v9: {n_loaded}')

In [ ]:
# Treina a melhor arquitetura encontrada com o protocolo final e mede desempenho em teste.
# O objetivo é confirmar se a arquitetura vencedora do estudo mantém vantagem num treino completo.
train_loader, val_loader, test_loader = get_loaders(BEST_BATCH_SIZE)
criterion  = nn.CrossEntropyLoss(label_smoothing=BEST_LABEL_SMOOTH)  # Loss final herdada do melhor setup anterior.
optimizer  = Adam(model_final.parameters(), lr=BEST_LR, weight_decay=BEST_WEIGHT_DECAY)  # Otimizador final da melhor arquitetura.
scheduler  = ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5, min_lr=1e-7)  # Baixa LR quando a validação estagna.
scaler     = torch.amp.GradScaler('cuda')

best_val_acc = 0.0
best_state   = None
history      = []

for epoch in range(EPOCHS_FINAL):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model_final, train_loader, optimizer, criterion, scaler)  # Aprende a melhor arquitetura em treino completo.
    val_loss,   val_acc   = evaluate(model_final, val_loader, criterion)  # Mede generalização após cada época.
    scheduler.step(val_acc)  # ReduceLROnPlateau usa a validação para ajustar o learning rate.
    elapsed = time.time() - t0

    history.append({'epoch': epoch, 'train_acc': round(train_acc,4),
                    'val_acc': round(val_acc,4), 'train_loss': round(train_loss,4),
                    'val_loss': round(val_loss,4)})

    improved = ' ✓' if val_acc > best_val_acc else ''
    print(f'[v10] {epoch+1:02d}/{EPOCHS_FINAL} | '
          f'Train: {train_loss:.4f} ({train_acc*100:.1f}%) | '
          f'Val: {val_loss:.4f} ({val_acc*100:.1f}%){improved} | {elapsed:.0f}s')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state   = deepcopy(model_final.state_dict())  # Guarda os pesos do ponto de melhor validação, não apenas os da última época.

# Avaliação final
model_final.load_state_dict(best_state)
_, test_acc = evaluate(model_final, test_loader, criterion)
print(f'\n✓ Treino final concluído!')
print(f'  Melhor val acc: {best_val_acc*100:.2f}%')
print(f'  Test acc:       {test_acc*100:.2f}%')
print(f'  vs v9 val:      {best_val_acc*100 - V9_VAL_ACC:+.2f}%')
print(f'  vs v9 test:     {test_acc*100 - V9_TEST_ACC:+.2f}%')

---
## 8. Guardar Resultados

In [ ]:
# Guarda checkpoint, métricas e metadados da arquitetura final para reutilização posterior.
# Assim o resultado do architecture search fica documentado e pronto para ser distribuído.
import pandas as pd
import subprocess

# ── Checkpoint do modelo ──────────────────────────────────────────────────────
ckpt_path = f"{PATHS['checkpoints']}/best_cnn_v10.pth"
torch.save({
    'model'         : best_state,
    'val_acc'       : best_val_acc,
    'test_acc'      : test_acc,
    'arch': {
        'conv_channels' : ch_best,
        'dense_sizes'   : ds_best,
        'embedding_size': emb_best,
        'dropout'       : do_best,
    },
    'history'       : history,
}, ckpt_path)
print(f'✓ Checkpoint guardado: {ckpt_path}')

# ── CSV — todos os trials ─────────────────────────────────────────────────────
records = []
for t in study.trials:
    if t.value is None:
        continue
    p  = t.params
    nc = p['n_conv_blocks']
    ch = [p[f'conv_ch_{i}'] for i in range(1, nc+1)]
    nd = p.get('n_dense_layers', 0)
    records.append({
        'trial'          : t.number,
        'val_acc'        : round(t.value, 4),
        'n_conv_blocks'  : nc,
        'conv_channels'  : str(ch),
        'n_dense_layers' : nd,
        'dense_sizes'    : str([p.get(f'dense_size_{i}') for i in range(1, nd+1)]),
        'embedding_size' : p.get('embedding_size'),
        'dropout'        : round(p.get('dropout', 0), 4),
    })
df = pd.DataFrame(records).sort_values('val_acc', ascending=False)
df.to_csv(f'{BASE}/arch_trials.csv', index=False)
print('✓ arch_trials.csv')

# ── CSV — histórico de treino ─────────────────────────────────────────────────
pd.DataFrame(history).to_csv(f'{BASE}/v10_training_history.csv', index=False)
print('✓ v10_training_history.csv')

# ── Publicar modelo no Kaggle ─────────────────────────────────────────────────
model_dir = f'{BASE}/upload_cnn_v10'
os.makedirs(model_dir, exist_ok=True)
shutil.copy(ckpt_path, f'{model_dir}/best_cnn_v10.pth')

result = subprocess.run([
    'kaggle', 'models', 'instances', 'versions', 'create',
    'goncalojesus/facecnn/pytorch/default',
    '-p', model_dir
], capture_output=True, text=True)
print(f'\nKaggle Model: {(result.stdout or result.stderr).strip()}')

print(f'\n── Resumo Final ─────────────────────────────────────────')
print(f'  Arquitetura: conv={ch_best} | dense={ds_best} | emb={emb_best}')
print(f'  Val acc:     {best_val_acc*100:.2f}% (v9: {V9_VAL_ACC}%)')
print(f'  Test acc:    {test_acc*100:.2f}% (v9: {V9_TEST_ACC}%)')